# 05 — Evidence-first research synthesis

## Scenario: recommend a retrieval strategy for Northstar Cloud

Engineering wants a short recommendation on hybrid retrieval and reranking. The answer must separate evidence from limitations, avoid overclaiming, and let a reviewer trace every claim to its sources. This deterministic lab models the workflow without an API key.

## The synthesis lifecycle

```text
decision question -> bounded evidence plan -> authorized retrieval
                                          -> deduplicate + inspect sources
                                          -> claim/evidence map
                                          -> findings + limitations + unknowns
                                          -> cited, reviewable recommendation
```

The model may help summarize; it does not decide source authority, access scope, or whether missing evidence is proof.

## 1. Create a small, diverse evidence set

In production, attach source metadata (publisher, date, version, authority, license, tenant scope, and exact supporting span). The fixture uses source IDs so every claim can be audited. Include both benefits and operational limitations rather than selecting only evidence that confirms the proposed solution.

In [ ]:
from src.rag_core.lesson_loader import load_lesson_module; globals().update({name: value for name, value in vars(load_lesson_module('curriculum/intermediate/05-research-synthesis/lab.py')).items() if not name.startswith('_')})

sources = [
    Document('retrieval-study', 'Hybrid retrieval preserves lexical identifiers while dense retrieval improves paraphrase matching.'),
    Document('reranking-study', 'Cross-encoder reranking can improve final evidence ordering on a bounded candidate set.'),
    Document('operations-note', 'Reranking can increase latency, cost, and timeout risk when candidate pools are unbounded.'),
    Document('support-data', 'Support incidents often contain exact error codes that need lexical fallback.'),
]
question = 'How should Northstar Cloud combine retrieval methods for support research?'
[(source.doc_id, source.text) for source in sources]

## 2. Plan evidence views, then retrieve and deduplicate

The query plan retains the original question and adds evidence, limitation, and operational views. Deduplication prevents a repeated source from dominating the context, but it is not proof of diversity: several sources can still repeat one upstream claim. Record the plan and returned IDs in a trace.

In [ ]:
plan = research_queries(question)
evidence = retrieve_unique(question, sources, top_k=3)
trace = trace_evidence(question, sources, top_k=3)
print('QUERY PLAN:')
for item in plan: print('-', item)
print('\nEVIDENCE IDs:', [source.doc_id for source in evidence])
print('TRACE:', trace)
assert trace.queries[0] == question
assert len(trace.source_ids) == len(set(trace.source_ids))

## 3. Build a claim–evidence map before writing

Claims should carry source IDs, a claim type, confidence, and limitations. The deterministic classifier flags terms such as cost/risk as limitations so the notebook shows the structure; production systems should use reviewed extraction or constrained schemas and require a reviewer for high-impact claims. A source citation does not automatically prove every nearby sentence.

In [ ]:
claims = make_claims(question, sources)
outline = synthesis_outline(claims)
for kind, rows in outline.items():
    print(f'\n{kind.upper()}')
    for claim in rows:
        print({'text': claim.text, 'sources': claim.source_ids, 'confidence': claim.confidence})

assert citation_coverage(claims) == 1.0
assert outline['limitation']

## 4. Draft a calibrated synthesis

A safe recommendation is narrow: use hybrid retrieval for exact identifiers and paraphrases, then consider reranking only after measuring candidate recall and p95 latency. The operational note qualifies the recommendation; it is not buried under a positive conclusion. If evidence for a target tenant, language, or workload is missing, label it as an open question and request evaluation rather than extrapolating.

Example claim-level citation style:

- Hybrid retrieval can preserve exact identifiers while improving paraphrase matching (`retrieval-study`, `support-data`).
- A bounded reranker may improve final ordering (`reranking-study`), but unbounded pools can harm latency and cost (`operations-note`).
- Recommendation: benchmark an authorized hybrid baseline, then add a bounded reranker only if the evaluation gate improves the relevant support slices.

## 5. Review and production checklist

- Validate identity and permissions before every research query.
- Preserve document/version/span IDs, query plan, deduplication decisions, claims, citations, and reviewer edits.
- Retrieve counterevidence and distinguish disagreement from absence.
- Do not generate references; only cite records returned by the evidence store.
- Use human review and a rubric for high-impact conclusions.
- Evaluate claim support, citation coverage, source diversity, abstention, latency, and cost.

### Exercises

1. Add two conflicting sources and write a claim that exposes their scope difference instead of choosing a winner.
2. Add a stale document; define whether it becomes a limitation, historical context, or a retrieval exclusion.
3. Add a tenant-restricted source and prove it never enters the evidence trace for an unauthorized user.
4. Create a reviewer rubric for source authority, claim support, counterevidence, and uncertainty.

### References

- [RAG paper](https://arxiv.org/abs/2005.11401)
- [Ragas paper](https://arxiv.org/abs/2309.15217)
- [NIST Generative AI Profile](https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.600-1.pdf)